# Fine-tuning LLM's with LoRa & QLoRa
<a target="_blank" href="https://colab.research.google.com/github/unionai/workshops/blob/main/tutorials/llm-fine-tuning-lora-qlora/llm-fine-tune-tutorial.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

This tutorial fine-tunes a small language model on text-to-SQL using full fine-tuning, LoRA, or QLoRA, and runs the whole pipeline on [Modal](https://modal.com).

Full code and repo:

https://github.com/unionai/workshops/tree/main/tutorials/llm-fine-tuning-lora-qlora

In [ ]:
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    # setup project if in colab env
    !git clone https://github.com/unionai/workshops
    %cd workshops/tutorials/llm-fine-tuning-lora-qlora
    !uv pip install -r requirements.txt

## Set up Modal (one-time)

Authenticate with Modal. This opens a browser to log in (sign up free at [modal.com](https://modal.com)).

The training and serving functions read `HF_TOKEN` from a Modal secret named `huggingface-secret` (needed for gated models, optional otherwise).

In [ ]:
!modal setup

# Create the HuggingFace secret (skip if you already have one):
# !modal secret create huggingface-secret HF_TOKEN=hf_...

## Run the fine-tuning pipeline

##### Pipeline Parameters

| Flag | Default | Description |
|------|---------|-------------|
| `--model-name` | `HuggingFaceTB/SmolLM2-135M` | HuggingFace model to fine-tune |
| `--dataset-name` | `b-mc2/sql-create-context` | HuggingFace dataset |
| `--method` | `lora` | Fine-tuning method: `full`, `lora`, or `qlora` |
| `--epochs` | `3` | Training epochs |
| `--lr` | `2e-4` | Learning rate |
| `--batch-size` | `4` | Per-device batch size |
| `--max-train-samples` | `5000` | Max training examples |
| `--max-eval-samples` | `500` | Max evaluation examples |
| `--num-eval-examples` | `50` | Examples for before/after comparison |
| `--lora-r` | `16` | LoRA rank (for lora/qlora) |
| `--lora-alpha` | `32` | LoRA alpha (for lora/qlora) |

In [ ]:
!modal run workflow.py --method lora --epochs 3 --max-train-samples 5000 --num-eval-examples 50

`modal run` executes the pipeline in the cloud on GPU. The fine-tuned model is saved to the shared `lora-qlora` volume, and the HTML reports (`training_report.html`, `evaluation_report.html`, `report.html`) are written to your working directory.

Watch live logs and run status in the [Modal dashboard](https://modal.com/apps) or with:

```
modal app list
```

## Serve the Model

Wait until the training pipeline is complete before serving the fine-tuned model. `serve.py` loads the model from the `lora-qlora` volume and exposes a FastAPI `/generate` endpoint.

Use `modal serve` for a live-reloading dev URL, or `modal deploy` for a persistent one.

In [ ]:
# Dev server (prints the endpoint URLs):
!modal serve serve.py

# Persistent deploy:
# !modal deploy serve.py

In [ ]:
# Replace the URL below with the /generate endpoint printed by `modal serve serve.py`
!curl -X POST https://your-workspace--finetuned-sql-api-model-generate.modal.run \
  -H "Content-Type: application/json" \
  -d '{
    "schema": "CREATE TABLE employees (id INT, name VARCHAR, department VARCHAR, salary INT)",
    "question": "What is the average salary by department?"
  }'

# Serve Gradio UI for Model

An interactive frontend that loads the fine-tuned model directly from the volume.

In [ ]:
!modal serve app_gradio.py

# Persistent deploy:
# !modal deploy app_gradio.py

Want to do more?

- Modal docs: https://modal.com/docs
- Modal examples: https://modal.com/docs/examples
- Modal dashboard: https://modal.com/apps